# Data engineering with Databricks - Real-time data ingestion for Financial transactions

Building a real-time system consuming messages from live systems is required to build reactive data application.

Near real-time is key to detecting new fraud patterns and to building a proactive system, offering better protection for your customers.

Ingesting, transforming and cleaning data to create clean SQL tables for our downstream users (Data Analysts and Data Scientists) is complex.

<link href="https://fonts.googleapis.com/css?family=DM Sans" rel="stylesheet"/>
<div style="width:300px; text-align: center; float: right; margin: 30px 60px 10px 10px;  font-family: 'DM Sans'">
  <div style="height: 300px; width: 300px;  display: table-cell; vertical-align: middle; border-radius: 50%; border: 25px solid #fcba33ff;">
    <div style="font-size: 70px;  color: #70c4ab; font-weight: bold">
      73%
    </div>
    <div style="color: #1b5162;padding: 0px 30px 0px 30px;">of enterprise data goes unused for analytics and decision making</div>
  </div>
  <div style="color: #bfbfbf; padding-top: 5px">Source: Forrester</div>
</div>

<br>

## <img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/de.png" style="float:left; margin: -35px 0px 0px 0px" width="80px"> John, as a Data engineer, spends an immense amount of time….


* Hand-coding data ingestion & transformations and dealing with technical challenges:<br>
  *Supporting streaming and batch, handling concurrent operations, small-file issues, GDPR requirements, complex DAG dependencies...*<br><br>
* Building custom frameworks to enforce quality and tests<br><br>
* Building and maintaining scalable infrastructure, with observability and monitoring<br><br>
* Managing incompatible governance models from different systems
<br style="clear: both">



<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F01-Data-ingestion%2F01.1-sdp-sql%2F01-SDP-fraud-detection-SQL&demo_name=lakehouse-fsi-fraud&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-fraud%2F01-Data-ingestion%2F01.1-sdp-sql%2F01-SDP-fraud-detection-SQL&version=1">

## Demo: build a banking database and detect fraud on transactions in real-time (ms)

In this demo, we'll step into the shoes of a retail banking company processing transactions.

The business has determined that we should improve our transaction fraud system and offer better protection to our customers (retail and institutions using our payment systems). We're asked to:

* Analyse and explain current transactions: quantify fraud, understand patterns and usage
* Build a proactive system to detect fraud and serve predictions in real-time (with ms latencies)


### What we'll build

To do so, we'll build an end-to-end solution with the Lakehouse. To be able to properly analyse and detect fraud, we'll mainly focus on transactional data, received by our banking system.

At a very high level, this is the flow we'll implement:

<img width="1000px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/lakehouse-fsi-fraud-overview-1.png" />

1. Ingest and create our banking database, with tables easy to query in SQL
2. Secure data and grant read access to the Data Analyst and Data Science teams.
3. Run BI queries to analyse existing fraud
4. Build an ML model to detect fraud and deploy this model for real-time inference

As a result, we'll have all the information required to trigger alerts and ask our customers for stronger authentication if we believe there is a high fraud risk.

**A note on Fraud detection in real applications** <br/>
*This demo is a simple example to showcase the Lakehouse benefits. We'll keep the data model and ML simple for the sake of the demo. Real-world applications would need more data sources and also deal with imbalanced classes and more advanced models. If you are interested in a more advanced discussion, reach out to your Databricks team!*

Let's see how this data can be used within the Lakehouse to analyse and reduce fraud!


## Building a Spark Declarative Pipelines pipeline to analyze and reduce fraud detection in real-time

In this example, we'll implement an end-to-end SDP pipeline consuming our banking transactions information. We'll use the medallion architecture but we could build a star schema, a data vault or any other modelisation.

We'll incrementally load new data with the Autoloader and enrich this information.

This information will then be used to:

* Build our DBSQL dashboard to track transactions and fraud impact.
* Train & deploy a model to detect potential fraud in real-time.

Let's implement the following flow:

<img width="1200px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-full.png"/>


*Note that we're including the ML model our [Data Scientist built]($../04-Data-Science-ML/04.1-AutoML-FSI-fraud) using Databricks AutoML to predict fraud. We'll cover that in the next section.*

Your SDP Pipeline has been installed and started for you! Open the <a dbdemos-pipeline-id="sdp-fsi-fraud" href="#joblist/pipelines/85846030-1b40-49ab-8061-01064a82269b" target="_blank">Fraud detection Spark Declarative Pipelines pipeline</a> to see it in action.<br/>
*(Note: The pipeline will automatically start once the initialization job is completed, this might take a few minutes... Check installation logs for more details)*


## 1/ Data Exploration

All Data projects start with some exploration. Open the [/explorations/sample_exploration]($./explorations/sample_exploration) notebook to get started and discover the data made available to you.

### 2/ Loading our data using Databricks Autoloader (cloud_files)

<img  style="float:right; margin-left: 10px" width="600px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-1.png"/>

Autoloader allows us to efficiently ingest millions of files from a cloud storage, and supports efficient schema inference and evolution at scale.

For more details on autoloader, run `dbdemos.install('auto-loader')`.

Let's use it to our pipeline and ingest the raw JSON & CSV data being delivered in our blob storage `/dbdemos/fsi/fraud-detection/...`.

Open the [/transformations/01-bronze.sql]($./transformations/01-bronze.sql) file to review the SQL code ingesting the raw data and creating our bronze layer.

### 3/ Enforce quality and materialize our tables for Data Analysts

<img style="float:right; margin-left: 10px" width="600px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-2.png"/>


The next layer, often called Silver, consumes **incremental** data from the bronze one, and cleans up some information:

* Clean up the codes of the countries of origin and destination (removing the "--")
* Calculate the difference between the Originating and Destination Balances.

We're also adding an [expectation](https://docs.databricks.com/workflows/delta-live-tables/delta-live-tables-expectations.html) on different fields to enforce and track our Data Quality. This will ensure that our dashboards are relevant and can easily spot potential errors due to data anomalies.

For more advanced SDP capabilities run `dbdemos.install('pipeline-bike')` or `dbdemos.install('declarative-pipeline-cdc')` for CDC/SCD Type 2 example.

These tables are clean and ready to be used by the BI team!

Open the [/transformations/02-silver.sql]($./transformations/02-silver.sql) file to review the SQL code creating our clean silver table.

### 4/ Aggregate and join data to create our ML features

<img style="float:right; margin-left: 10px" width="600px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-3.png"/>

We're now ready to create the features required for Fraud detection.

We need to enrich our transaction dataset with extra information that our model will use to help detect fraud.

Open the [/transformations/03-gold.sql]($./transformations/03-gold.sql) file to review the SQL code creating our enriched gold table.

## Our pipeline is now ready!

As you can see, building data pipelines with databricks lets you focus on your business implementation while the engine solves all hard data engineering work for you.

The table is now ready for our Data Scientist to train a model detecting fraud risk.

Open the <a dbdemos-pipeline-id="sdp-fsi-fraud" href="#joblist/pipelines/85846030-1b40-49ab-8061-01064a82269b" target="_blank">Fraud detection Spark Declarative Pipelines pipeline</a> and click on Start to visualize your lineage and consume the new data incrementally!

# Next: secure and share data with Unity Catalog

Now that these tables are available in our Lakehouse, let's review how we can share them with the Data Scientists and Data Analysts teams.

Jump to the [Governance with Unity Catalog notebook]($../../02-Data-governance/02-UC-data-governance-ACL-fsi-fraud) or [Go back to the introduction]($../../00-FSI-fraud-detection-introduction-lakehouse)



### Source Data

This dataset is built with PaySim, an open source banking transactions simulator.

[PaySim](https://github.com/EdgarLopezPhD/PaySim) simulates mobile money transactions based on a sample of real transactions extracted from one month of financial logs from a mobile money service implemented in an African country.